In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import sys
from pathlib import Path
import os
import hoomd
import gsd.hoomd
import matplotlib.pyplot as plt
from pathlib import Path

validation_dir = Path("/groups/goloborodko/projects/leona/validation")
bulk_dir = validation_dir / "bulk"
output_dir = bulk_dir / "output_dir"
sys.path.extend([str(bulk_dir), str(validation_dir)])

analysis_dir = Path("/groups/goloborodko/projects/leona/validation/analysis")
if str(analysis_dir) not in sys.path:
	sys.path.insert(0, str(analysis_dir))

In [40]:
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.graph_objects as go

from src.plots.diagnostic import compute_projection_2d, plot_projection_2d, compute_rdf, plot_rdf, average_rdf
from src.plots.panels import plot_panels
from src.utils.io import *

In [35]:
fig_root = validation_dir / "analysis" / "analysis_output" / "plots" / "diagnostic_plots_from_traj"

# --- settings ---
SWEEP         = "50b_reentrant_full"
TRAJ_NAME     = "trajectory_0.gsd"
PLANE         = "xy"
BINS          = 300
SIGMA         = 0
AVERAGE       = False
FRACTION_LAST = 0.2
N_COLS        = 3

sweep_dir = output_dir / SWEEP
fig_dir   = fig_root / SWEEP
fig_dir.mkdir(parents=True, exist_ok=True)

In [36]:
def plot_one_projection(run_dir: Path, ax):
    with gsd.hoomd.open(run_dir / TRAJ_NAME, "r") as traj:
        H, ea, eb = compute_projection_2d(
            traj, plane=PLANE, bins=BINS, sigma=SIGMA,
            average=AVERAGE, fraction_last=FRACTION_LAST,
        )
    plot_projection_2d(H, ea, eb, ax=ax, plane=PLANE, cbar=False)

In [37]:
PLANES        = ["xy", "xz", "yz"]
META_KEYS     = ["E_att", "angle_k_ppp", "molecule_file", "density"]



run_dirs = [p for p in sorted(sweep_dir.iterdir())
            if p.is_dir() and (p / TRAJ_NAME).exists()]
print(f"{len(run_dirs)} runs with {TRAJ_NAME}")

AXES = {"x": 0, "y": 1, "z": 2}

for run_dir in run_dirs:
    params = load_params_json(run_dir / "params.json")
    meta = " | ".join(f"{k}={params.get(k)}" for k in META_KEYS)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    with as_traj(run_dir / TRAJ_NAME) as traj:
        # row 1: 2D projections
        for ax, plane in zip(axes[0], PLANES):
            H, ea, eb = compute_projection_2d(
                traj, plane=plane, bins=BINS, sigma=SIGMA,
                average=AVERAGE, fraction_last=FRACTION_LAST,
            )
            plot_projection_2d(H, ea, eb, ax=ax, plane=plane)

        # row 2: 1D count histograms along x, y, z (averaged over last frames)
        frames = get_frames(traj, FRACTION_LAST) if AVERAGE else [len(traj) - 1]
        for ax, (name, i) in zip(axes[1], AXES.items()):
            L = traj[frames[-1]].configuration.box[i]
            edges = np.linspace(-L/2, L/2, BINS + 1)
            counts = np.zeros(BINS)
            for fi in frames:
                pos = np.asarray(traj[fi].particles.position)[:, i]
                counts += np.histogram(pos, bins=edges)[0]
            # NOTE: no /= len(frames)  -> summed counts like before
            centers = 0.5 * (edges[:-1] + edges[1:])
            ax.fill_between(centers, counts, step="mid", alpha=0.4)
            ax.plot(centers, counts, drawstyle="steps-mid")
            ax.set_xlabel(name); ax.set_ylabel("counts")
            ax.set_ylim(bottom=0)

    fig.suptitle(f"{run_dir.name}\n{meta}", fontsize=9)
    fig.tight_layout()
    out = fig_dir / f"{run_dir.name}__last_frame_projection.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("saved →", out)


48 runs with trajectory_0.gsd
saved → /groups/goloborodko/projects/leona/validation/analysis/analysis_output/plots/diagnostic_plots_from_traj/50b_reentrant_full/0_E_att_1_angle_k_ppp_0_density_0.01_molecule_file_cag47x100.json_N_steps_40000000_N_write_10000__last_frame_projection.png
saved → /groups/goloborodko/projects/leona/validation/analysis/analysis_output/plots/diagnostic_plots_from_traj/50b_reentrant_full/10_E_att_2_angle_k_ppp_5_density_0.01_molecule_file_cag47x100.json_N_steps_40000000_N_write_10000__last_frame_projection.png
saved → /groups/goloborodko/projects/leona/validation/analysis/analysis_output/plots/diagnostic_plots_from_traj/50b_reentrant_full/11_E_att_2_angle_k_ppp_5_density_0.005_molecule_file_cag47x100.json_N_steps_40000000_N_write_10000__last_frame_projection.png
saved → /groups/goloborodko/projects/leona/validation/analysis/analysis_output/plots/diagnostic_plots_from_traj/50b_reentrant_full/12_E_att_4_angle_k_ppp_0_density_0.01_molecule_file_cag47x100.json_N_st

In [41]:
from collections import defaultdict

R_MAX  = None      # None → 0.95 * Lmin/2
N_BINS = 100

data = defaultdict(dict)  # {density: {(angle, E): (r, g)}}

for run_dir in run_dirs:
    p = load_params_json(run_dir / "params.json")
    key = (p["angle_k_ppp"], p["E_att"])
    with as_traj(run_dir / TRAJ_NAME) as traj:
        if AVERAGE:
            frames = get_frames(traj, FRACTION_LAST)
            r, g = average_rdf(traj, frames=frames, bins=N_BINS, r_max=R_MAX)
        else:
            r, g = compute_rdf(traj[-1], bins=N_BINS, r_max=R_MAX)
    data[p["density"]][key] = (r, g)

for density, entries in data.items():
    angles = sorted({k[0] for k in entries})
    eatts  = sorted({k[1] for k in entries})
    nR, nC = len(angles), len(eatts)

    fig, axes = plt.subplots(nR, nC, figsize=(3*nC, 2.5*nR),
                             sharex=True, sharey=True, squeeze=False)
    for i, a in enumerate(angles):
        for j, e in enumerate(eatts):
            ax = axes[i, j]
            if (a, e) in entries:
                r, g = entries[(a, e)]
                plot_rdf(r, g, ax=ax, annotate=False)
            if i == 0: ax.set_title(f"E_att={e}", fontsize=9)
            if j == 0: ax.set_ylabel(f"angle={a}", fontsize=9)

    fig.suptitle(f"{SWEEP} — RDF | density={density}", fontsize=11)
    fig.tight_layout()
    out = fig_dir / f"{SWEEP}__rdf__density{density}.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("saved →", out)

saved → /groups/goloborodko/projects/leona/validation/analysis/analysis_output/plots/diagnostic_plots_from_traj/50b_reentrant_full/50b_reentrant_full__rdf__density0.01.png
saved → /groups/goloborodko/projects/leona/validation/analysis/analysis_output/plots/diagnostic_plots_from_traj/50b_reentrant_full/50b_reentrant_full__rdf__density0.005.png


In [42]:
from collections import defaultdict

data = defaultdict(dict)  # {density: {(angle, E): (H, ea, eb)}}

for run_dir in run_dirs:
    p = load_params_json(run_dir / "params.json")
    key = (p["angle_k_ppp"], p["E_att"])
    with as_traj(run_dir / TRAJ_NAME) as traj:
        H, ea, eb = compute_projection_2d(traj[-1], plane="xy", bins=BINS, sigma=SIGMA)
    data[p["density"]][key] = (H, ea, eb)

for density, entries in data.items():
    angles = sorted({k[0] for k in entries})
    eatts  = sorted({k[1] for k in entries})
    nR, nC = len(angles), len(eatts)

    fig, axes = plt.subplots(nR, nC, figsize=(3*nC, 3*nR), squeeze=False)
    for i, a in enumerate(angles):
        for j, e in enumerate(eatts):
            ax = axes[i, j]
            if (a, e) in entries:
                H, ea_, eb_ = entries[(a, e)]
                plot_projection_2d(H, ea_, eb_, ax=ax, plane="xy", cbar=False)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_xlabel(""); ax.set_ylabel("")
            if i == 0: ax.set_title(f"E_att={e}", fontsize=9)
            if j == 0: ax.set_ylabel(f"angle={a}", fontsize=9)

    fig.suptitle(f"{SWEEP} — xy projection (last frame) | density={density}", fontsize=11)
    fig.tight_layout()
    out = fig_dir / f"{SWEEP}__projection_xy_lastframe__density{density}.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("saved →", out)

saved → /groups/goloborodko/projects/leona/validation/analysis/analysis_output/plots/diagnostic_plots_from_traj/50b_reentrant_full/50b_reentrant_full__projection_xy_lastframe__density0.01.png
saved → /groups/goloborodko/projects/leona/validation/analysis/analysis_output/plots/diagnostic_plots_from_traj/50b_reentrant_full/50b_reentrant_full__projection_xy_lastframe__density0.005.png
